<a href="https://colab.research.google.com/github/Rahul15072006/Data_science_assignment/blob/main/International_T20_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv('International_T20_Data.csv')

In [3]:
df.head()

,innings,meta.data_version,meta.created,meta.revision,info.dates,info.gender,info.match_type,info.outcome.by.wickets,info.outcome.winner,info.overs,...,info.outcome.by.runs,info.match_type_number,info.neutral_venue,info.outcome.method,info.outcome.result,info.outcome.eliminator,info.supersubs.New Zealand,info.supersubs.South Africa,info.bowl_out,info.outcome.bowl_out
0,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-18,2,"[datetime.date(2017, 2, 17)]",male,T20,5.0,Sri Lanka,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-19,2,"[datetime.date(2017, 2, 19)]",male,T20,2.0,Sri Lanka,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-23,1,"[datetime.date(2017, 2, 22)]",male,T20,NaN,Australia,20,...,41.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"[{'1st innings': {'team': 'Hong Kong', 'delive...",0.9,2016-09-12,1,"[datetime.date(2016, 9, 5)]",male,T20,NaN,Hong Kong,20,...,40.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"[{'1st innings': {'team': 'Zimbabwe', 'deliver...",0.9,2016-06-19,1,"[datetime.date(2016, 6, 18)]",male,T20,NaN,Zimbabwe,20,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Rename all the column names to their appropriate names, for example meta.created should be renamed as created_date

In [10]:
df = df.rename(columns={
    "meta.created": "created_date",
    "meta.data_version": "data_version",
    "meta.revision": "revision_number",
    "info.city": "city",
    "info.venue": "venue",
    "info.dates": "match_date",
    "info.gender": "gender",
    "info.match_type": "match_type",
    "info.teams": "teams",
    "info.toss.winner": "toss_winner",
    "info.toss.decision": "toss_decision",
    "info.outcome.winner": "match_winner",
    "info.outcome.by.runs": "win_by_runs",
    "info.outcome.by.wickets": "win_by_wickets"
}, errors="ignore")


# Find out the top three venues which hosted the greatest number of matches.

In [11]:
top_venues = df['venue'].value_counts().head(3)

print(top_venues)

venue
Dubai International Cricket Stadium    62
Sheikh Zayed Stadium                   41
Shere Bangla National Stadium          39
Name: count, dtype: int64


## Find out the pair of cricket teams who played the most number of T20 matches against each other.

In [14]:
df['team_pair'] = df['teams'].apply(lambda x: tuple(sorted(x)))

# Count how many times each pair appears
team_pair_counts = df['team_pair'].value_counts()

# Get the pair that played the most matches
most_matches_pair = team_pair_counts.head(1)

print(most_matches_pair)

# Count matches played by each team
matches_played = {}

for teams in df['teams']:
    for team in teams:
        matches_played[team] = matches_played.get(team, 0) + 1

# Count matches won by each team
matches_won = df['match_winner'].value_counts()

# Create dataframe for calculation
win_stats = pd.DataFrame({
    'matches_played': pd.Series(matches_played),
    'matches_won': matches_won
}).fillna(0)

# Calculate win percentage
win_stats['win_percentage'] = (win_stats['matches_won'] / win_stats['matches_played']) * 100

# Sort and get top 5 teams
top_5_teams = win_stats.sort_values(by='win_percentage', ascending=False).head(5)

print(top_5_teams)


team_pair
( , ', ', ', ', ,, A, E, [, ], a, a, a, d, g, i, l, l, n, n, r, s, t, u)    45
Name: count, dtype: int64
             matches_played  matches_won  win_percentage
Australia               0.0        132.0             inf
Bahrain                 0.0          1.0             inf
Afghanistan             0.0         51.0             inf
Bulgaria                0.0          1.0             inf
Botswana                0.0          2.0             inf


## Print the top five teams by their win percentages. Win percentage is defined as the number of matches won divided by the number of matches played and then multiplied by 100.

In [19]:
import pandas as pd
import ast

df = pd.read_csv("International_T20_Data.csv")
df['info.teams'] = df['info.teams'].apply(ast.literal_eval)
matches_played = df['info.teams'].explode().value_counts()
matches_won = df['info.outcome.winner'].value_counts()
stats = pd.DataFrame({
    'Matches_Played': matches_played,
    'Matches_Won': matches_won
}).fillna(0)

stats['Win_Percentage'] = (stats['Matches_Won'] / stats['Matches_Played']) * 100
top_5 = stats.sort_values('Win_Percentage', ascending=False).head(5)
print(top_5)

             Matches_Played  Matches_Won  Win_Percentage
Belgium                   3          3.0      100.000000
Spain                     6          5.0       83.333333
Germany                  17         13.0       76.470588
Namibia                  34         25.0       73.529412
Afghanistan              75         51.0       68.000000


## Write a function to get the scorecard of each match. This function would take the innings value as argument and return two scorecard dataframes each for one team as shown below. So the first dataframe would contain the top 4 scorers of the team who batted first and the top 4 bowlers of the opponent team. And the second dataframe would contain the top 4 scorers of the team who batted second and the top 4 bowlers of the opponent team.

In [22]:
import pandas as pd
import ast
df = pd.read_csv("International_T20_Data.csv")
df["innings"] = df["innings"].apply(ast.literal_eval)

def get_scorecard(innings):
    result = []

    for inn in innings:
        deliveries = list(inn.values())[0]["deliveries"]
        bat, bowl = {}, {}

        for ball in deliveries:
            info = list(ball.values())[0]
            bat[info["batsman"]] = bat.get(info["batsman"], 0) + info["runs"]["batsman"]
            b = info["bowler"]
            bowl.setdefault(b, {"runs":0, "wkts":0})
            bowl[b]["runs"] += info["runs"]["total"]
            if "wicket" in info:
                bowl[b]["wkts"] += 1

        bat_df = pd.DataFrame(bat.items(), columns=["Player","Runs"])\
                   .sort_values("Runs", ascending=False).head(4)
        bowl_df = pd.DataFrame(
            [(k,v["wkts"],v["runs"]) for k,v in bowl.items()],
            columns=["Bowler","Wickets","Runs_Conceded"]
        ).sort_values(["Wickets","Runs_Conceded"],
                      ascending=[False,True]).head(4)

        result.append((bat_df, bowl_df))

    return result
    score = get_scorecard(df["innings"].iloc[0])
score = get_scorecard(df["innings"].iloc[0])
# First innings
print(score[0][0])  # Top 4 batters
print(score[0][1])  # Top 4 bowlers

#second innings
print(score[1][0])
print(score[1][1])


      Player  Runs
0   AJ Finch    43
1  M Klinger    38
2    TM Head    31
4  AJ Turner    18
           Bowler  Wickets  Runs_Conceded
0      SL Malinga        2             29
5   DAS Gunaratne        1             11
4  PADLR Sandakan        1             31
2   JRMVB Sanjaya        1             35
            Player  Runs
3    DAS Gunaratne    52
2   EMDY Munaweera    44
0      N Dickwella    30
4  TAM Siriwardana    15
        Bowler  Wickets  Runs_Conceded
5    AJ Turner        2             12
4      A Zampa        2             26
0   PJ Cummins        1             30
2  JP Faulkner        0             29
